# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

All entities are referenced by their `@id` for clarity and reproducibility. Let's inspect the available record sets and their fields.

In [ ]:
from pprint import pprint

# Access record sets
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the dataset.")
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f"Record Set: {rs.name} (@id: {rs.id})")
        print("Fields:")
        for field in rs.fields:
            print(f"  - {field.name} (@id: {field.id}, type: {getattr(field, 'data_type', 'N/A')})")
        print('-' * 40)

# For further exploration, pick the first record set
if record_sets:
    main_record_set_id = record_sets[0].id
    print(f"Main record set selected for extraction: {main_record_set_id}")
else:
    main_record_set_id = None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 
Use record set and field `@id`s as determined above.

In [ ]:
# If no record sets found, stop
if not record_sets:
    print("No record sets available; cannot extract records.")
else:
    # List all available record set @id's
    record_set_ids = [rs.id for rs in record_sets]
    print("Extracting data from record set(s):", record_set_ids)
    dataframes = {}
    for record_set_id in record_set_ids:
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        if not records:
            print(f"No records found for {record_set_id}")
            continue
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"First five columns in {record_set_id}:\n{df.columns[:5].tolist()}")
        display(df.head())

    first_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

In [ ]:
import numpy as np

# Ensure data has been loaded
if not record_sets or not dataframes:
    print("No data available for EDA.")
else:
    # Pick the main record set for EDA
    df = dataframes[first_record_set_id]

    # Show numeric columns (they should have type float, int, etc.)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields available for analysis: {numeric_cols}")

    # Attempt to select a valid numeric field; else, skip
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 10
        print(f"Using numeric field: {numeric_field_id} with threshold: {threshold}")
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, normalized_col]].head())

        # Try grouping by a non-numeric field, e.g., the first object-type field
        group_field = None
        object_cols = df.select_dtypes(include=["object"]).columns.tolist()
        if object_cols:
            group_field = object_cols[0]
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No string/object fields to group by.")
    else:
        print("No numeric fields to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize distribution of the selected numeric field
if 'df' in locals() and not df.empty and 'numeric_field_id' in locals():
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'group_field' in locals() and group_field:
        # Boxplot for grouped data
        plt.figure(figsize=(12,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR⁲ mass spectrometry dataset using the `mlcroissant` library. We:

- Loaded and described the dataset and its schema.
- Examined available record sets and their `@id` fields.
- Loaded data into pandas DataFrames via record set `@id`s.
- Conducted basic EDA, such as filtering and normalizing a numeric field, and grouped by a categorical field.
- Visualized data distributions.

This structure can be extended to more advanced analyses or to other Croissant-compliant datasets as needed.